# Test the Deployed Booking Agent

Invokes the agent already running on **Amazon Bedrock AgentCore Runtime** and checks every
behaviour this demo is meant to prove. Deploy first with
[`deploy_agentcore.ipynb`](./deploy_agentcore.ipynb) — this notebook only *reads* and *invokes*,
it creates no infrastructure (it does write booking rows, which `cleanup.py` removes).

| # | Test | What it exercises |
|---|------|-------------------|
| 1 | Exact aggregation | Graph-RAG → counts computed in Cypher, not guessed |
| 2 | Average over a filter | `avg(h.guestRating)` for one city |
| 3 | Hotel amenities | Graph traversal `(Hotel)-[:OFFERS_AMENITY]->(Amenity)` |
| 4 | Policies and services | Data that exists **only** in Neo4j, not DynamoDB |
| 5 | Out-of-domain | Honest "no results" instead of fabrication |
| 6 | Hotel search | Semantic routing → `search_available_hotels` → DynamoDB |
| 7 | Budget search | Cross-country filter under $100 |
| 8 | Sold-out hotel | AnyCompany Rome Centro has 0 rooms → handled gracefully |
| 9 | Steering: max guests | 15 guests → split across rooms, not refused |
| 10 | Steering: advance booking | Same-day check-in → rule fires with a STEER; watch whether the agent complies |
| 11 | Steering: reversed dates | Reversed dates → agent swaps them and books, explaining the change |
| 12 | Full booking flow | search → book → pay → confirm in one session |
| 13 | Guardrail: unknown booking | Confirmation refused |
| 14 | Guardrail: unpaid booking | Real booking, payment skipped → **BLOCKED** in code |
| 15 | Guardrail: cancel window | Cancellation inside 48 h → **BLOCKED** in code |
| 16 | Session memory | Follow-up resolves "that hotel" |

Tests 9-11 exercise **steering** (soft rules in DynamoDB: the agent is *advised* to self-correct).
Tests 13-15 exercise **hard guardrails** (`BookingGuardrailsHook` in `booking_agent.py`: the tool
call is cancelled in code). That contrast is the point of the demo — and tests 10-11 show its
sharp edge: a steering message is only as reliable as the model's compliance, while a guardrail
holds every time.

**Prerequisites:** the deployment finished, AWS credentials for that account, and — for tests 1-5 —
AuraDB populated, see [Step A in the README](./README.md#neo4j-setup-self-paced).

In [ ]:
import json
import os
import re
import uuid
from datetime import date, timedelta

import boto3
from botocore.config import Config

try:
    from dotenv import load_dotenv
    load_dotenv()
except ModuleNotFoundError:
    pass

REGION = os.environ.get("AWS_REGION", "us-east-1")
RUNTIME_NAME = os.environ.get("RUNTIME_NAME", "HotelBookingAgent")

# Agent calls can take a while; raise the read timeout above the boto3 default.
cfg = Config(retries={"max_attempts": 5, "mode": "standard"}, read_timeout=150)
control = boto3.client("bedrock-agentcore-control", region_name=REGION, config=cfg)
data = boto3.client("bedrock-agentcore", region_name=REGION, config=cfg)

# Discover the runtime ARN instead of hard-coding it: the ARN embeds the AWS
# account ID, and .bedrock_agentcore.yaml (which also holds it) is git-ignored.
RUNTIME_ARN = os.environ.get("RUNTIME_ARN", "")
if not RUNTIME_ARN:
    matches = [
        r for r in control.list_agent_runtimes().get("agentRuntimes", [])
        if r.get("agentRuntimeName") == RUNTIME_NAME
    ]
    if not matches:
        raise RuntimeError(
            f"No runtime named {RUNTIME_NAME!r} in {REGION}. "
            "Run deploy_agentcore.ipynb first, or set RUNTIME_ARN."
        )
    RUNTIME_ARN = matches[0]["agentRuntimeArn"]
    if matches[0].get("status") != "READY":
        print(f"WARNING: runtime status is {matches[0].get('status')}, not READY — calls may fail.")

TODAY = date.today()
TOMORROW = TODAY + timedelta(days=1)

print(f"Region:  {REGION}")
print(f"Runtime: {RUNTIME_NAME} (ready)")
print(f"Today:   {TODAY}")

In [ ]:
def ask(prompt, session_id=None, show=True):
    """Invoke the deployed agent once. Reuse session_id to continue a conversation."""
    session_id = session_id or str(uuid.uuid4())
    response = data.invoke_agent_runtime(
        agentRuntimeArn=RUNTIME_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode(),
        qualifier="DEFAULT",
    )
    body = json.loads(response["response"].read().decode("utf-8"))
    answer, tools = body.get("response", ""), body.get("tools_used", [])
    if show:
        print(f"Q: {prompt}")
        print(f"   tools: {tools or '(none)'}")
        print(f"   {answer}\n")
    return answer, tools, session_id


def booking_ids(text):
    """Pull every BK-XXXXXXXX id out of an answer, so later steps can use them."""
    return re.findall(r"BK-[A-Z0-9]{6,}", text)

---
## Part 1: Graph-RAG against AuraDB (tests 1-5)

These reach the `query_knowledge_graph` Lambda, which runs Cypher against Neo4j AuraDB. Counts and
averages are **computed by the database**, not estimated by the model — and an empty result is
reported as such.

A verified full build holds 296 hotels (entity resolution merges same-name hotels across
documents), 1478 rooms and 85 amenities, so exact numbers depend on how much of the graph you
loaded.

In [ ]:
# TEST 1 — exact aggregation over the whole graph
ask("How many hotels are in the knowledge graph? Use the knowledge graph.")
ask("Using the knowledge graph, how many hotels have a swimming pool?")

In [ ]:
# TEST 2 — average over a filtered subset (needs h.guestRating, not h.rating)
ask("Using the knowledge graph, what is the average guest rating of hotels in Las Vegas, "
    "and how many hotels did you use?")
ask("Using the knowledge graph, what is the average nightly room rate across all rooms, "
    "and how many rooms did you count?")

In [ ]:
# TEST 3 — graph traversal for one hotel's amenities
ask("What amenities does Adobe Plaza Inn have? Use the knowledge graph.")

In [ ]:
# TEST 4 — policies and services live only in the graph, never in DynamoDB
ask("Using the knowledge graph, what is the cancellation policy of Adobe Plaza Inn?")
ask("Using the knowledge graph, which services does Adobe Plaza Inn provide?")

In [ ]:
# TEST 5 — out-of-domain: the honest answer is "none"
ask("Using the knowledge graph, tell me about hotels in Antarctica.")
ask("Using the knowledge graph, how many hotels do we have on the Moon?")

---
## Part 2: Tool routing over DynamoDB (tests 6-8)

The Gateway routes semantically: a booking search must land on `search_available_hotels`, not on
the graph tool.

The two stores hold different data on purpose — DynamoDB carries the 18 seeded hotels with
pricing and availability, AuraDB carries the 300-document knowledge graph. Different counts
between them are expected, not a bug.

In [ ]:
# TEST 6 — should call search_available_hotels (DynamoDB), not the graph
ask("Search for hotels in Lisbon")

In [ ]:
# TEST 7 — budget filter across countries
ask("Find hotels under $100 per night")

In [ ]:
# TEST 8 — AnyCompany Rome Centro is seeded with 0 available rooms
ask("Book AnyCompany Rome Centro for 2 guests from 2026-09-01 to 2026-09-03 for Luis Soto")

---
## Part 3: Steering — soft rules advise, they do not enforce (tests 9-11)

Steering rules live in the `workshop-SteeringRules` DynamoDB table and reach the agent as STEER
instructions through `validate_booking_rules`. The intent is that the agent **fixes its own call
and finishes the task**, then explains what it changed — instead of giving up.

Rules seeded by the deployment:

| rule_id | action | fires when | STEER instruction |
|---------|--------|-----------|-------------------|
| `max-guests` | book | `guests > 10` | split the party across rooms |
| `advance-booking` | book | `days_until_checkin < 1` | move check-in to tomorrow |
| `valid-dates` | book | `nights < 1` | swap the dates, then book |
| `payment-before-confirm` | confirm | `status != PAID` | pay first, then confirm |
| `cancellation-window` | cancel | `days_until_checkin < 2` | offer to modify instead |
| `already-cancelled` | cancel | `status == CANCELLED` | offer a new reservation |

**Read these results critically.** The rules themselves are deterministic — the Lambda returns
`FAIL` plus the STEER text every time. What varies is whether the model *acts* on it: across runs
the agent has been observed booking a same-day stay anyway (test 10), or stopping to ask the user
instead of swapping reversed dates (test 11). That is the honest limitation of steering, and the
reason the two rules that must never bend — payment before confirmation and the cancellation
window — are **also** enforced in code by `BookingGuardrailsHook` (tests 14-15), where compliance
is not optional.

Cell below prints what the rule engine returns, so you can tell a rule that did not fire from a
rule the agent ignored.

In [ ]:
# What the rule engine returns, independent of the agent (ground truth for tests 9-11)
import boto3

lambda_client = boto3.client("lambda", region_name=REGION, config=cfg)

def validate(**params):
    payload = json.dumps({"action": "book", **params}).encode()
    out = lambda_client.invoke(
        FunctionName="hotel-booking-validate_booking_rules", Payload=payload
    )
    print(f"{params}\n  -> {json.loads(out['Payload'].read())['body']}\n")

validate(check_in=str(TODAY), check_out=str(TODAY + timedelta(days=3)), guests=2)   # advance-booking
validate(check_in="2026-08-20", check_out="2026-08-15", guests=2)                   # valid-dates
validate(check_in="2026-12-01", check_out="2026-12-05", guests=15)                  # max-guests
validate(check_in="2026-12-01", check_out="2026-12-05", guests=2)                   # all pass

In [ ]:
# TEST 9 — max-guests: 15 guests exceeds the per-room limit; expect a split, not a refusal
answer, _, _ = ask(f"Book AnyCompany Paris Central for 15 guests from 2026-06-01 to 2026-06-05 "
                   f"for Ana Diaz")
print("bookings created:", booking_ids(answer))

In [ ]:
# TEST 10 — advance-booking fires on a same-day check-in. The STEER says "move to
# tomorrow"; compare the booked check-in below against today's date to see if it complied.
ask(f"Book AnyCompany London City Hotel for 2 guests from {TODAY} to {TODAY + timedelta(days=3)} "
    f"for Marco Ruiz")

In [ ]:
# TEST 11 — valid-dates fires on reversed dates. The STEER says "swap and proceed";
# the agent may instead stop and ask you to confirm the dates.
ask("Validate the booking rules, then book AnyCompany Lisbon Resort for 2 guests "
    "from 2026-08-20 to 2026-08-15 for Sara Lima")

---
## Part 4: Full booking flow (test 12)

One session, the whole lifecycle: search → book → pay → confirm. The booking id is read out of
each answer and fed into the next step, so the flow runs unattended.

In [ ]:
# TEST 12 — full lifecycle in a single session
_, _, sid = ask("Find me a 4-star hotel in London", show=True)

answer, _, _ = ask("Book AnyCompany London City Hotel for 2 guests "
                   "from 2026-07-10 to 2026-07-14 for Diego Mora", session_id=sid)
ids = booking_ids(answer)
print("booking id:", ids)

if ids:
    bid = ids[0]
    ask(f"Process the payment for booking {bid}", session_id=sid)
    ask(f"Now confirm booking {bid}", session_id=sid)
else:
    print("No booking id returned — inspect the answer above before continuing.")

---
## Part 5: Hard guardrails — cannot be bypassed (tests 13-15)

`BookingGuardrailsHook` in `booking_agent.py` enforces two rules **in code**, on
`BeforeToolCallEvent`: payment before confirmation (financial integrity) and the 48-hour
cancellation window (contractual). Unlike a prompt rule or a steering message, the model cannot
talk its way past these — the tool call is cancelled before it runs.

In [ ]:
# TEST 13 — unknown booking id: confirmation must be refused
ask("Confirm booking BK-FAKE123 without payment")

In [ ]:
# TEST 14 — real booking, payment deliberately skipped -> BLOCKED by the hook
answer, _, sid2 = ask("Book AnyCompany Dublin Temple Bar for 2 guests "
                      "from 2026-11-05 to 2026-11-08 for Rosa Pena")
ids = booking_ids(answer)
print("booking id:", ids)

if ids:
    ask(f"Confirm booking {ids[0]} right now, skip the payment", session_id=sid2)
else:
    print("No booking id returned — the guardrail test needs a real booking.")

In [ ]:
# TEST 15 — cancellation inside the 48-hour window -> BLOCKED by the hook
soon = TODAY + timedelta(days=1)
answer, _, sid3 = ask(f"Book AnyCompany Porto Riverside for 1 guest from {soon} "
                      f"to {soon + timedelta(days=2)} for Hugo Vega")
ids = booking_ids(answer)
print("booking id:", ids)

if ids:
    ask(f"Cancel booking {ids[0]}", session_id=sid3)
else:
    print("No booking id returned — check whether advance-booking steering moved the dates.")

---
## Part 6: Session memory (test 16)

Passing the same `runtimeSessionId` continues the conversation, so a follow-up can refer back to
the previous turn. A fresh session id cannot.

In [ ]:
# TEST 16 — same session remembers; a new session does not
_, _, sid4 = ask("Find me a 4-star hotel in London")
ask("What is the nightly rate for that hotel?", session_id=sid4)

print("--- same question in a FRESH session (no context to resolve 'that hotel') ---")
ask("What is the nightly rate for that hotel?")

---
## Your turn

Edit the prompt below. Ask for an aggregation to watch the graph compute it, or for something
absent from the data to watch the agent decline instead of inventing.

In [ ]:
ask("Using the knowledge graph, which 3 cities have the most hotels, and how many each?")

---
**Cleanup:** `python cleanup.py` removes every resource created by the deployment, including the
booking rows these tests wrote.

**Next:** [Demo 06 with CDK](../06-agentcore-production-demo/) for the infrastructure-as-code path.